# 03 — Feature Engineering
### IndicNews AI — Hindi News Analysis, Retrieval & Recommendation System

Builds the TF-IDF feature representation: one fitted vectorizer, reused everywhere. Also
compares against Bag-of-Words 


In [4]:
pip install scipy scikit-learn

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ------------- -------------------------- 2.9/8.3 MB 18.6 MB/s eta 0:00:01
   ------------------------------ --------- 6.3/8.3 MB 16.1 MB/s eta 0:00:01
   ---------------------------------------  8.1/8.3 MB 15.2 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 13.2 MB/s  0:00:00
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ------------- -------------------------- 1/3 [narwhals]
   ------------- -------------------------- 1/3 [narwhals]
   ------------- -------------------------- 1/3 [narwhals]
   ------------- -------------------------- 1/3 [narwhals]
   ------------- -------------------------- 1/3 [narwhals]
   ------------- -------------------------- 1/3 [narwhals]
   ------------- -------------------------- 1/3 [narwhals]
   ------------- -------------------------- 1/3 [narwhals]
   ------------- ---------

In [5]:
import sys
sys.path.append("..")

import time
import numpy as np
import pandas as pd
import joblib
import scipy.sparse as sp

from data_utils import load_dataset
from preprocessing import preprocess_pipeline
from feature_engineering import (
    build_tfidf_vectorizer, build_count_vectorizer,
    build_combined_token_field, finalize_tokens_for_features,
    EXTENDED_STOPWORDS_HI,
)

pd.set_option("display.max_colwidth", 80)

In [6]:
df = load_dataset(verbose=False)
uniq = df.drop_duplicates(subset=["Headline", "Content"]).reset_index(drop=True)
clean_df = preprocess_pipeline(uniq, columns=["Headline", "Content"])
print(clean_df.shape)

(34826, 11)


## 1. Why NOT a raw document-frequency stopword cutoff

"the highest-document-frequency stemmed tokens" an extra stopword layer. Testing that idea directly
against the real corpus shows it's too blunt:

In [7]:
from collections import Counter
doc_freq = Counter()
for toks in clean_df["Content_tokens"]:
    doc_freq.update(set(toks))

n_docs = len(clean_df)
top_df = (pd.Series(doc_freq).sort_values(ascending=False) / n_docs * 100).round(1)
top_df.head(20)

उन्हों     24.7
ले         18.8
बत         17.9
गई         16.1
रह         15.6
बन         14.9
जा         14.9
लग         14.1
लोग        13.2
गए         12.6
बकौल       12.0
मिल        11.8
भारत       11.5
मुताबिक    11.3
दी         11.1
दे         11.1
वीडिय      10.8
भारतीय     10.4
2          10.1
2024        9.9
dtype: float64

`भारत` (India, 11.5%) and `भारतीय` (Indian, 10.4%) sit right next to
genuine function words like `उन्हों`/`ले`/`बत`. A raw frequency cutoff
would delete real content words along with the filler which is bad for both
classification and keyword extraction. 
`EXTENDED_STOPWORDS_HI`
in `feature_engineering.py` is a small, *manually reviewed* set 
reporting/auxiliary-verb stems (`ले`, `बत`, `गई`, `बन`, `जा`...) and
attribution connectors (`मुताबिक`, `बकौल`, `दौरान`) not "whatever is
frequent". Content words like `भारत`/`पुलिस`/`भारतीय` are deliberately
kept; TF-IDF's own IDF term downweights universally-common words without
deleting them outright, which is the textbook-correct tool for that.

In [8]:
print(sorted(EXTENDED_STOPWORDS_HI))
print("count:", len(EXTENDED_STOPWORDS_HI))

['उन्हों', 'कर', 'गई', 'गए', 'चल', 'जा', 'दी', 'दे', 'दौरान', 'बकौल', 'बत', 'बन', 'मिल', 'मुताबिक', 'रह', 'लग', 'लिख', 'ले', 'साम']
count: 19


## 2. Building the TF-IDF feature matrix

 `build_combined_token_field()` concatenates each row's
`Headline_tokens` + `Content_tokens`, applies `EXTENDED_STOPWORDS_HI`,
then feeds the result to `build_tfidf_vectorizer()` — unigrams through
trigrams, `min_df=5`, `max_df=0.85` (drop terms in >85% of docs : a
straightforward, defensible way to catch near-universal terms without
the manual-curation debate).

In [12]:
combined_tokens = build_combined_token_field(clean_df)
combined_tokens.iloc[0][:15]

['कांग्रेस',
 'ने',
 'बलजिंदर',
 'सिंह',
 'पंजाब',
 'गोल',
 'मार',
 'हत्य',
 'कांग्रेस',
 'ने',
 'बलजिंदर',
 'सिंह',
 'सोमवार',
 'पंजाब',
 'मोग']

In [13]:
t0 = time.time()
tfidf = build_tfidf_vectorizer()
X_tfidf = tfidf.fit_transform(combined_tokens)
print(f"Fit + transform: {time.time() - t0:.1f}s")
print("Matrix shape:", X_tfidf.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))
sparsity = X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1])
print(f"Sparsity: {sparsity:.4%} nonzero ({X_tfidf.nnz:,} nonzero entries)")

Fit + transform: 8.6s
Matrix shape: (34826, 60359)
Vocabulary size: 60359
Sparsity: 0.0772% nonzero (1,622,958 nonzero entries)


In [14]:
feat_names = tfidf.get_feature_names_out()
unigrams = sum(1 for f in feat_names if " " not in f)
bigrams = sum(1 for f in feat_names if f.count(" ") == 1)
trigrams = sum(1 for f in feat_names if f.count(" ") == 2)
print("Vocab breakdown -> unigrams:", unigrams, " bigrams:", bigrams, " trigrams:", trigrams)

Vocab breakdown -> unigrams: 12593  bigrams: 35144  trigrams: 12622


## 3. Bag-of-Words comparison


In [15]:
t0 = time.time()
bow = build_count_vectorizer(ngram_range=(1, 1))
X_bow = bow.fit_transform(combined_tokens)
print(f"BoW (unigrams) fit_transform: {time.time() - t0:.2f}s")
print("BoW vocabulary (unigrams only):", len(bow.vocabulary_))

BoW (unigrams) fit_transform: 1.06s
BoW vocabulary (unigrams only): 12593


Same unigram vocabulary size as TF-IDF's unigram slice (both built from
the same tokens/thresholds) the difference is entirely in *weighting*,
not vocabulary: BoW gives every occurrence equal weight, TF-IDF
downweights terms that are common across most documents.

## 4. Top terms by mean TF-IDF weight

In [16]:
mean_tfidf = np.asarray(X_tfidf.mean(axis=0)).ravel()
top_idx = mean_tfidf.argsort()[::-1][:15]
for i in top_idx:
    print(f"{feat_names[i]:15s} {mean_tfidf[i]:.4f}")

लोग             0.0099
भारत            0.0095
वीडिय           0.0093
पुलिस           0.0079
भारतीय          0.0078
शेयर            0.0077
2               0.0076
महिल            0.0076
चुनाव           0.0075
मैं             0.0069
साल             0.0069
दिल्ल           0.0068
आरोप            0.0067
वाल             0.0066
सरकार           0.0065


In [17]:
bigram_idx = [i for i, f in enumerate(feat_names) if f.count(" ") == 1]
trigram_idx = [i for i, f in enumerate(feat_names) if f.count(" ") == 2]
top_bi = sorted(bigram_idx, key=lambda i: -mean_tfidf[i])[:10]
top_tri = sorted(trigram_idx, key=lambda i: -mean_tfidf[i])[:10]

print("Top bigrams:")
for i in top_bi:
    print(f"  {feat_names[i]:25s} {mean_tfidf[i]:.4f}")
print("\nTop trigrams:")
for i in top_tri:
    print(f"  {feat_names[i]:35s} {mean_tfidf[i]:.4f}")

Top bigrams:
  विश्व कप                  0.0043
  लोकसभ चुनाव               0.0038
  उत्तर प्रदेश              0.0036
  नरेंद्र मोद               0.0035
  प्रधानमंत्र नरेंद्र       0.0033
  सोशल मीडिय                0.0032
  वीडिय आय                  0.0028
  टी20 विश्व                0.0027
  पीएम मोद                  0.0026
  वीडिय वायरल               0.0022

Top trigrams:
  प्रधानमंत्र नरेंद्र मोद             0.0033
  टी20 विश्व कप                       0.0027
  विश्व कप 2024                       0.0017
  लोकसभ चुनाव 2024                    0.0016
  वनड विश्व कप                        0.0015
  वीडिय सोशल मीडिय                    0.0013
  सोशल मीडिय वायरल                    0.0013
  विश्व कप 2023                       0.0012
  मुख्यमंत्र अरविंद केजरीवाल          0.0012
  जिसक वीडिय आय                       0.0010


The bigrams/trigrams are genuinely meaningful named entities and
phrases (`प्रधानमंत्र नरेंद्र मोद`, `विश्व कप`, `लोकसभ चुनाव`)  exactly
the kind of multi-word signal unigram-only BoW/TF-IDF would miss.

## 5. Persisting the fitted vectorizer



In [ ]:
joblib.dump(tfidf, "../../models/saved_models/tfidf_vectorizer.joblib")
sp.save_npz("../../models/saved_models/tfidf_matrix.npz", X_tfidf)


reloaded = joblib.load("../../models/saved_models/tfidf_vectorizer.joblib")
X_check = reloaded.transform(combined_tokens[:5])
print("Reload OK, shape:", X_check.shape)

import os
vec_mb = os.path.getsize("../../models/saved_models/tfidf_vectorizer.joblib") / 1e6
mat_mb = os.path.getsize("../../models/saved_models/tfidf_matrix.npz") / 1e6
print(f"Vectorizer: {vec_mb:.1f} MB   Matrix: {mat_mb:.1f} MB")

Reload OK, shape: (5, 60359)
Vectorizer: 2.5 MB   Matrix: 14.6 MB


## 8. Summary & decisions carried into later modules

| Finding | Decision for later modules |
|---|---|
| Raw document-frequency stopword cutoff deletes real content words (`भारत`, `पुलिस`) | Use a small, manually reviewed `EXTENDED_STOPWORDS_HI` instead; let TF-IDF's IDF handle the rest |
| scikit-learn's default tokenizer breaks Devanagari (matras aren't `\w`) | Feed pre-tokenized lists directly (`tokenizer=_identity, token_pattern=None`) — never let scikit-learn tokenize Hindi text itself |
| Lambda tokenizers aren't picklable | Use named module-level functions so the fitted vectorizer can be saved/reloaded |
| TF-IDF (1,3-gram): 60,359 features, 0.077% dense | Feasible as-is for Module 6 (Logistic Regression/SVM handle sparse input natively) |
| Bigrams/trigrams surface real entities (`विश्व कप`, `प्रधानमंत्र नरेंद्र मोद`) | Keep n-gram range (1,3) — don't drop to unigram-only |
| Vectorizer + matrix persisted to `models/saved_models/` (2.5 MB + 14.6 MB) | Module 6 loads this directly rather than refitting |

